# リプレイ分析ノートブック

Kaggle の「Download replay」で取得した JSON を読み込み、試合経過を人間が読める形式で表示する。

In [6]:
import sys
sys.path.insert(0, '../agent')
sys.path.insert(0, '../scripts')

import replay_utils as ru
from cg.api import SelectContext

# ---- 設定 ----
REPLAY_FILE = 'logs/85453660.json'
MY_NAME     = 'ok method'
CARD_CSV    = '../docs/JP_Card_Data.csv'

# --- 読み込み処理 ---
card_names = ru.load_card_names(CARD_CSV)
print('カードDB読み込み完了:', len(card_names), '件')

replay = ru.load_replay(REPLAY_FILE, MY_NAME)
my_idx, op_idx = replay.my_idx, replay.op_idx
steps, winner  = replay.steps, replay.winner
ctx_map, opt_map, area_map = replay.ctx_map, replay.opt_map, replay.area_map
print('リプレイ読み込み完了:', REPLAY_FILE ,len(steps), 'ステップ')
print(f'- 自分: {replay.my_name} (P{my_idx}, {"先攻" if my_idx == 0 else "後攻"})')
print(f'- 相手: {replay.op_name} (P{op_idx})')
print(f'- 勝者: {winner}')
print(f'- 総ステップ数: {len(steps)}')

カードDB読み込み完了: 1267 件
リプレイ読み込み完了: logs/85453660.json 71 ステップ
- 自分: ok method (P1, 後攻)
- 相手: Muhammad Ibrahim Qasmi (P0)
- 勝者: Muhammad Ibrahim Qasmi
- 総ステップ数: 71


In [2]:
# ---- サイド推移サマリー ----

print('=== サイド推移 ===')
prev_my   = None  # None = まだ初期化前
prev_op   = None
prev_turn = -1

for step in steps:
    obs = step[my_idx].get('observation', {})
    cur = obs.get('current')
    if not cur: continue
    turn = cur.get('turn', 0)
    ps   = cur.get('players', [])
    if len(ps) < 2: continue
    my_prize = len(ps[my_idx].get('prize') or [])
    op_prize = len(ps[op_idx].get('prize') or [])

    # 双方のサイドが揃った時点で初期化
    if prev_my is None:
        if my_prize == 6 and op_prize == 6:
            prev_my = 6; prev_op = 6
        continue

    my_took = prev_op - op_prize   # 相手サイド↓ = 自分が取った
    op_took = prev_my - my_prize   # 自分サイド↓ = 相手が取った

    if (my_took > 0 or op_took > 0) and turn != prev_turn:
        events = []
        if my_took > 0: events.append(f'自分+{my_took}枚')
        if op_took > 0: events.append(f'相手+{op_took}枚')
        print(f'  T{turn:2d}: 自{my_prize}枚 / 相{op_prize}枚  ({" / ".join(events)})')
        prev_my   = my_prize
        prev_op   = op_prize
        prev_turn = turn

print(f'\n最終結果: 【{winner}】の勝ち')

=== サイド推移 ===
  T 5: 自6枚 / 相5枚  (自分+1枚)

最終結果: 【Muhammad Ibrahim Qasmi】の勝ち


In [3]:
# ---- 自分の行動一覧（action が空でないステップのみ）----

print('=== 自分の選択一覧 ===')
prev_turn = -1

for step in steps:
    my_data = step[my_idx]
    action  = my_data.get('action')
    if not action:
        continue

    obs  = my_data.get('observation', {})
    cur  = obs.get('current')
    sel  = obs.get('select')
    if not cur or not sel:
        continue

    turn     = cur.get('turn', 0)
    context  = sel.get('context')
    options  = sel.get('option', []) or []
    ctx_name = ctx_map.get(context, str(context))

    if turn != prev_turn:
        print(f'\n--- T{turn} ---')
        prev_turn = turn

    chosen = [options[idx] for idx in action if idx < len(options)]
    for opt in chosen:
        otype = opt_map.get(opt.get('type'), str(opt.get('type')))
        area  = opt.get('area')
        pidx  = opt.get('playerIndex')
        oidx  = opt.get('index')

        if otype == 'PLAY':
            hand = ((cur.get('players') or [])[my_idx].get('hand', []))
            card = hand[oidx] if oidx is not None and oidx < len(hand) else None
        else:
            card = ru.get_card_from_obs(obs, area, oidx, pidx, area_map) if area is not None and oidx is not None else None

        card_info = f' [{ru.fmt_card(card, card_names)}]' if card else ''
        in_area   = area_map.get(opt.get('inPlayArea'), '')
        in_idx    = opt.get('inPlayIndex')
        target    = f' → {in_area}[{in_idx}]' if in_area else ''
        attack_id = opt.get('attackId')
        atk_info  = f' attackId={attack_id}' if attack_id else ''
        print(f'  [{ctx_name}] {otype}{card_info}{target}{atk_info}')

=== 自分の選択一覧 ===

--- T0 ---
  [SETUP_ACTIVE_POKEMON] CARD [ルナトーン]
  [DRAW_COUNT] NUMBER

--- T2 ---
  [MAIN] PLAY [パワープロテイン]
  [MAIN] END

--- T4 ---
  [MAIN] ATTACH → BENCH[0]
  [MAIN] PLAY [ポケモンいれかえ]
  [MAIN] END

--- T6 ---
  [MAIN] ATTACH → ACTIVE[0]
  [MAIN] PLAY [パワープロテイン]
  [MAIN] PLAY [パワープロテイン]
  [MAIN] ATTACK attackId=976
  [MAIN] ATTACK attackId=976


In [4]:
# ---- ターン別盤面詳細（MAINフェーズ開始時）----

print('=== ターン別盤面 ===')
seen_turns = set()

for step in steps:
    obs = step[my_idx].get('observation', {})
    cur = obs.get('current')
    sel = obs.get('select')
    if not cur or not sel:
        continue
    turn    = cur.get('turn', 0)
    context = sel.get('context')
    if context != int(SelectContext.MAIN):
        continue
    if turn in seen_turns:
        continue
    seen_turns.add(turn)

    print(f'\n=== T{turn} MAINフェーズ開始時 ===')
    ru.print_board(obs, my_idx, op_idx, card_names, area_map)

=== ターン別盤面 ===

=== T2 MAINフェーズ開始時 ===
  【相手】 サイド残6枚
    バトル場: マクノシタ HP80/80 E×1
    ベンチ  : ['ルナトーン HP110/110 E×0', 'ソルロック HP110/110 E×0', 'リオル HP80/80 E×0', 'リオル HP80/80 E×0']
  【自分】 サイド残6枚  手札8枚
    バトル場: ルナトーン HP110/110 E×0
    ベンチ  : []
    手札    : ['ポケモンいれかえ', '基本【闘】エネルギー', '基本【闘】エネルギー', '基本【闘】エネルギー', 'パワープロテイン', 'ポケモンいれかえ', '基本【闘】エネルギー', 'メガルカリオex']

=== T4 MAINフェーズ開始時 ===
  【相手】 サイド残6枚
    バトル場: マクノシタ HP80/80 E×2
    ベンチ  : ['ルナトーン HP110/110 E×0', 'ソルロック HP110/110 E×0', 'メガルカリオex HP340/340 E×0', 'リオル HP80/80 E×0']
  【自分】 サイド残6枚  手札8枚
    バトル場: ルナトーン HP100/100 E×1
    ベンチ  : []
    手札    : ['ポケモンいれかえ', '基本【闘】エネルギー', '基本【闘】エネルギー', 'パワープロテイン', 'ポケモンいれかえ', '基本【闘】エネルギー', 'メガルカリオex', 'マクノシタ']

=== T6 MAINフェーズ開始時 ===
  【相手】 サイド残5枚
    バトル場: メガルカリオex HP440/440 E×1
    ベンチ  : ['ルナトーン HP110/110 E×0', 'ソルロック HP110/110 E×0', 'マクノシタ HP80/80 E×1', 'メガルカリオex HP340/340 E×2', 'マクノシタ HP80/80 E×0']
  【自分】 サイド残6枚  手札7枚
    バトル場: マクノシタ HP80/80 E×1
    ベンチ  : []
    手札    : ['ポケモンいれかえ', '基本【闘】エネルギー',